# 🏆 Does the Golden Cross REALLY Work?
### BacktestLAB — Scientific Backtest

---

## 🗺️ Road Map

| # | Section                     | Goal                                            |
|---|-----------------------------|-------------------------------------------------|
| 1 | Setup & Data                | Load price data from Yahoo Finance              |
| 2 | Golden Cross on S&P 500     | Single asset, Death Cross vs TP/SL 15/5         |
| 3 | Multi-Asset Study           | Same test across 50 assets              |
| 4 | EUR/USD — A Fairer Test     | Apply strategy to a range-bound asset           |
| 5 | MA Optimisation             | Which MA combo works best on EUR/USD?           |
 Scientific Verdict          | Final conclusions                               |

> ⚠️ Disclaimer — Educational purposes only. Not financial advice.


---
## ⚙️ SECTION 1 — Setup & Data

In [ ]:
# ════════════════════════════════════════════════════════════════
# SECTION 1 — Setup & Data
# ════════════════════════════════════════════════════════════════

import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded")


✅ Libraries loaded


In [ ]:
# ════════════════════════════════════════════════════════════════
# SECTION 1 — Data Loading
# ════════════════════════════════════════════════════════════════

def getDataFromYahoo(symbol, start, end, frequency='1d', auto_adjust=True):
    """
    Download OHLCV data from Yahoo Finance.
    """
    try:
        data = yf.download(symbol, start=start, end=end,
                           interval=frequency, auto_adjust=auto_adjust,
                           progress=False)
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)
        df = data.reset_index()[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']]
        df.columns = ['datetime', 'open', 'high', 'low', 'close', 'volume']
        df['datetime'] = pd.to_datetime(df['datetime']).dt.tz_localize(None)
        return df.reset_index(drop=True)
    except Exception as e:
        print(f"  ⚠️  Error fetching {symbol}: {e}")
        return None


# ── Load demo asset (S&P 500, 20 years) ─────────────────────────
SYMBOL_SP = '^GSPC'
#symbol = 'EURUSD=X'
period = 365 * 20

start_20y = (datetime.datetime.now() - datetime.timedelta(days=period)).strftime('%Y-%m-%d')
end_date  = datetime.datetime.now().strftime('%Y-%m-%d')

df_sp = getDataFromYahoo(SYMBOL_SP, start_20y, end_date)

print(f'\n✅ {SYMBOL_SP}')
print(f'   Rows   : {len(df_sp)}')
print(f'   Period : {df_sp["datetime"].iloc[0].date()} → {df_sp["datetime"].iloc[-1].date()}')
df_sp.tail(3)


✅ ^GSPC
   Rows   : 5028
   Period : 2006-04-17 → 2026-04-10


,datetime,open,high,low,close,volume
5025,2026-04-08,6754.359863,6793.500000,6740.279785,6782.810059,5904880000
5026,2026-04-09,6783.689941,6835.310059,6761.549805,6824.660156,4912410000
5027,2026-04-10,6839.240234,6845.770020,6808.459961,6816.890137,4393220000


---
## 📈 SECTION 2 — Golden Cross on S&P 500

We apply the strategy to the S&P 500 over **20 years**.
Two exit strategies are compared: **Death Cross** and **TP 15% / SL 5%**.

| Event | Condition | Action |
|-------|-----------|--------|
| 🟡 **Golden Cross** | MA50 crosses **above** MA200 | **BUY** |
| ⚫ **Death Cross**  | MA50 crosses **below** MA200 | **SELL** |


In [ ]:
# ════════════════════════════════════════════════════════════════
# Signal Computation
# ════════════════════════════════════════════════════════════════

def compute_signals(df, short_w=50, long_w=200):
    """
    Add MA columns and Golden/Death Cross boolean flags.
    """
    df    = df.copy()
    col_s = f'ma{short_w}'
    col_l = f'ma{long_w}'

    df[col_s] = df['close'].rolling(short_w).mean()
    df[col_l] = df['close'].rolling(long_w).mean()

    df['signal']       = (df[col_s] > df[col_l]).astype(int)
    df['prev_signal']  = df['signal'].shift(1)
    df['golden_cross'] = (df['signal'] == 1) & (df['prev_signal'] == 0)
    df['death_cross']  = (df['signal'] == 0) & (df['prev_signal'] == 1)

    df.attrs['short_w'] = short_w
    df.attrs['long_w']  = long_w

    return df.dropna(subset=[col_s, col_l]).reset_index(drop=True)


In [ ]:

df_sp = compute_signals(df_sp, short_w=50, long_w=200)

print(f'   Golden Cross signals : {df_sp["golden_cross"].sum()}')
print(f'   Death  Cross signals : {df_sp["death_cross"].sum()}')

   Golden Cross signals : 10
   Death  Cross signals : 9


In [ ]:
# ── Visualise signals ────────────────────────────────────────────
def plot_signals(df, symbol):
    """
    Plot signals
    """
    short_w = df.attrs.get('short_w', 50)
    long_w  = df.attrs.get('long_w',  200)

    fig = go.Figure()
    fig.add_trace(go.Candlestick(
        x=df['datetime'], open=df['open'], high=df['high'],
        low=df['low'], close=df['close'], name='Price',
        increasing_line_color='#26a69a', decreasing_line_color='#ef5350'
    ))
    fig.add_trace(go.Scatter(x=df['datetime'], y=df[f'ma{short_w}'],
        line=dict(color='royalblue', width=1.5), name=f'MA {short_w}'))
    fig.add_trace(go.Scatter(x=df['datetime'], y=df[f'ma{long_w}'],
        line=dict(color='orange', width=3), name=f'MA {long_w}'))

    gc_df = df[df['golden_cross']]
    fig.add_trace(go.Scatter(x=gc_df['datetime'], y=gc_df['close'], mode='markers',
        marker=dict(symbol='triangle-up', size=13, color='gold',
                    line=dict(color='black', width=1)), name='Golden Cross ▲'))
    dc_df = df[df['death_cross']]
    fig.add_trace(go.Scatter(x=dc_df['datetime'], y=dc_df['close'], mode='markers',
        marker=dict(symbol='triangle-down', size=13, color='silver',
                    line=dict(color='red', width=1)), name='Death Cross ▼'))

    fig.update_layout(
        height=600, template='plotly_dark',
        title=dict(text=f'BackTestLAB — {symbol}  |  MA{short_w} vs MA{long_w}',
                   font=dict(size=17, color='gold')),
        xaxis_rangeslider_visible=False,
        legend=dict(orientation='h', y=1.02)
    )
    fig.update_xaxes(rangebreaks=[dict(bounds=['sat', 'mon'])], tickformat='%Y')
    fig.show()

plot_signals(df_sp, SYMBOL_SP)


In [ ]:
# ════════════════════════════════════════════════════════════════
# Trade Detection — Exit 1: Death Cross
# ════════════════════════════════════════════════════════════════

def detect_golden_cross_trades(df):
    """
    Entry : Golden Cross (MA_short crosses above MA_long)
    Exit  : Death Cross  (MA_short crosses below MA_long) or end of data
    """
    trades   = []
    in_trade = False

    for idx in df.index:
        if df.loc[idx, 'golden_cross'] and not in_trade:
            in_trade    = True
            entry_date  = df.loc[idx, 'datetime']
            entry_price = df.loc[idx, 'close']

        elif df.loc[idx, 'death_cross'] and in_trade:
            trades.append(_make_trade(
                entry_date, entry_price,
                df.loc[idx, 'datetime'], df.loc[idx, 'close'], 'closed'
            ))
            in_trade = False

    if in_trade:
        trades.append(_make_trade(
            entry_date, entry_price,
            df.loc[df.index[-1], 'datetime'], df.loc[df.index[-1], 'close'], 'open'
        ))
    return trades


    def _make_trade(entry_date, entry_price, exit_date, exit_price, outcome):
    """Helper to build a trade dict."""
    pnl = (exit_price - entry_price) / entry_price * 100
    return {
        'entry_date':   pd.Timestamp(entry_date),
        'entry_price':  round(float(entry_price), 5),
        'exit_date':    pd.Timestamp(exit_date),
        'exit_price':   round(float(exit_price),  5),
        'pnl_pct':      round(pnl, 2),
        'holding_days': (pd.Timestamp(exit_date) - pd.Timestamp(entry_date)).days,
        'outcome':      outcome,
    }

In [ ]:
# ════════════════════════════════════════════════════════════════
# Trade Detection — Exit 2: Take Profit / Stop Loss
# ════════════════════════════════════════════════════════════════

def detect_trades_tp_sl(df, tp=0.15, sl=0.05):
    """
    Same Golden Cross entry, exits at TP or SL.

    tp : float — take-profit threshold (e.g. 0.15 = +15%)
    sl : float — stop-loss threshold   (e.g. 0.05 =  -5%)
    """
    trades   = []
    in_trade = False

    highs  = df['high'].values
    lows   = df['low'].values
    closes = df['close'].values
    dates  = df['datetime'].values

    for i in range(len(df)):
        if df.iloc[i]['golden_cross'] and not in_trade:
            in_trade    = True
            entry_price = closes[i]
            entry_date  = dates[i]
            continue

        if in_trade:
            tp_level = entry_price * (1 + tp)
            sl_level = entry_price * (1 - sl)

            hit_tp = highs[i] >= tp_level
            hit_sl = lows[i]  <= sl_level

            if hit_tp or hit_sl:
                # SL takes priority when both triggered on the same bar
                if hit_sl:
                    exit_price = sl_level
                    outcome    = f'SL -{int(sl * 100)}%'
                else:
                    exit_price = tp_level
                    outcome    = f'TP +{int(tp * 100)}%'

                trades.append(_make_trade(
                    entry_date, entry_price, dates[i], exit_price, outcome
                ))
                in_trade = False

    if in_trade:
        trades.append(_make_trade(
            entry_date, entry_price,
            dates[-1], closes[-1], 'EOD'
        ))
    return trades



In [ ]:
# ── Run backtest ─────────────────────────────────────────────────
trades_sp_cross = detect_golden_cross_trades(df_sp)
trades_sp_tpsl  = detect_trades_tp_sl(df_sp, tp=0.15, sl=0.05)

print(f'Golden Cross → Death Cross   : {len(trades_sp_cross)} trades')
print(f'Golden Cross → TP 15%/SL 5%  : {len(trades_sp_tpsl)} trades')


Golden Cross → Death Cross   : 10 trades
Golden Cross → TP 15%/SL 5%  : 10 trades


In [ ]:
# ════════════════════════════════════════════════════════════════
# Performance Metrics
# ════════════════════════════════════════════════════════════════

def compute_bh_return(df):
    """Buy-and-Hold total return + CAGR over the full data window."""
    bh    = (df['close'].iloc[-1] / df['close'].iloc[0] - 1) * 100
    n_yrs = (df['datetime'].iloc[-1] - df['datetime'].iloc[0]).days / 365
    cagr  = ((1 + bh / 100) ** (1 / n_yrs) - 1) * 100 if n_yrs > 0 else 0
    return round(bh, 1), round(cagr, 2)


def compute_metrics(trades_list, df, label='Strategy'):
    """
    Compute performance metrics from a list of trade dicts.
    Returns are compounded trade-by-trade.
    Sharpe is annualised via the average holding period.
    """
    if not trades_list:
        return {}

    rets = np.array([t['pnl_pct'] / 100 for t in trades_list])

    equity      = np.cumprod(1 + rets)
    total_ret   = round((equity[-1] - 1) * 100, 1)

    rolling_max = np.maximum.accumulate(equity)
    max_dd      = round(((equity - rolling_max) / rolling_max).min() * 100, 1)

    n_years  = (df['datetime'].iloc[-1] - df['datetime'].iloc[0]).days / 365
    cagr     = round(((1 + total_ret / 100) ** (1 / n_years) - 1) * 100, 2) if n_years > 0 else 0

    avg_hold   = np.mean([t['holding_days'] for t in trades_list])
    ann_factor = np.sqrt(max(252 / avg_hold, 1)) if avg_hold > 0 else 1
    sharpe     = round((np.mean(rets) / np.std(rets)) * ann_factor, 2) if np.std(rets) > 0 else 0.0

    bh_ret, _ = compute_bh_return(df)

    return {
        'label':        label,
        'n_trades':     len(trades_list),
        'win_rate':     round(np.mean(rets > 0) * 100, 1),
        'avg_trade':    round(np.mean(rets) * 100, 2),
        'total_ret':    total_ret,
        'bh_ret':       bh_ret,
        'cagr':         cagr,
        'sharpe':       sharpe,
        'max_dd':       max_dd,
        'avg_holding':  round(avg_hold, 0),
        'equity_curve': equity,
        'exit_dates':   [t['exit_date'] for t in trades_list],
    }


def display_trades(trades, label=""):
    """Colour-coded trade log table."""
    t_df = pd.DataFrame(trades)[[
        'entry_date', 'entry_price', 'exit_date', 'exit_price',
        'pnl_pct', 'holding_days', 'outcome'
    ]].copy()
    t_df['entry_date'] = pd.to_datetime(t_df['entry_date']).dt.strftime('%Y-%m-%d')
    t_df['exit_date']  = pd.to_datetime(t_df['exit_date']).dt.strftime('%Y-%m-%d')

    def color_row(row):
        bg = '#1a3a1a; color: #b5f5a0' if row['pnl_pct'] > 0 else '#3a1a1a; color: #f5a0a0'
        return [f'background-color: {bg}'] * len(row)

    print(f'\n── {label} ──')
    return t_df.style.apply(color_row, axis=1).format({'pnl_pct': '{:+.2f}%'})




print("✅ All functions loaded")


✅ All functions loaded


In [ ]:

m_sp_cross = compute_metrics(trades_sp_cross, df_sp, 'Death Cross')
m_sp_tpsl  = compute_metrics(trades_sp_tpsl,  df_sp, 'TP 15% / SL 5%')
display(display_trades(trades_sp_cross, 'EXIT: Death Cross'))
display(display_trades(trades_sp_tpsl,  'EXIT: TP 15% / SL 5%'))


── EXIT: Death Cross ──


,entry_date,entry_price,exit_date,exit_price,pnl_pct,holding_days,outcome
0,2007-01-31,1438.239990,2007-12-21,1484.459960,+3.21%,324,closed
1,2009-06-23,895.099980,2010-07-02,1022.580020,+14.24%,374,closed
2,2010-10-22,1183.079960,2011-08-12,1178.810060,-0.36%,294,closed
3,2012-01-31,1312.410030,2015-08-28,1988.870000,+51.54%,1305,closed
4,2015-12-21,2021.150020,2016-01-11,1923.670040,-4.82%,21,closed
5,2016-04-25,2087.790040,2018-12-07,2633.080080,+26.12%,956,closed
6,2019-04-01,2867.189940,2020-03-30,2626.649900,-8.39%,364,closed
7,2020-07-09,3152.050050,2022-03-14,4173.109860,+32.39%,613,closed
8,2023-02-02,4179.759770,2025-04-14,5405.970210,+29.34%,802,closed
9,2025-07-01,6198.009770,2026-04-10,6816.890140,+9.99%,283,open



── EXIT: TP 15% / SL 5% ──


,entry_date,entry_price,exit_date,exit_price,pnl_pct,holding_days,outcome
0,2007-01-31,1438.239990,2007-03-14,1366.327990,-5.00%,42,SL -5%
1,2009-06-23,895.099980,2009-08-24,1029.364970,+15.00%,62,TP +15%
2,2010-10-22,1183.079960,2011-04-28,1360.541950,+15.00%,188,TP +15%
3,2012-01-31,1312.410030,2013-01-29,1509.271540,+15.00%,364,TP +15%
4,2015-12-21,2021.150020,2016-01-08,1920.092520,-5.00%,18,SL -5%
5,2016-04-25,2087.790040,2017-03-01,2400.958540,+15.00%,310,TP +15%
6,2019-04-01,2867.189940,2020-01-15,3297.268430,+15.00%,289,TP +15%
7,2020-07-09,3152.050050,2020-11-09,3624.857560,+15.00%,123,TP +15%
8,2023-02-02,4179.759770,2023-02-23,3970.771780,-5.00%,21,SL -5%
9,2025-07-01,6198.009770,2026-04-10,6816.890140,+9.99%,283,EOD


In [ ]:
# ── Metrics summary table ────────────────────────────────────────
metrics_sp = pd.DataFrame([
    {
        'Exit Strategy': m['label'],
        'Trades':        m['n_trades'],
        'Win Rate %':    m['win_rate'],
        'Avg Trade %':   m['avg_trade'],
        'Total Ret %':   m['total_ret'],
        'B&H Ret %':     m['bh_ret'],
        'CAGR %':        m['cagr'],
        'Sharpe':        m['sharpe'],
        'Max DD %':      m['max_dd'],
        'Avg Hold (d)':  m['avg_holding'],
    }
    for m in [m_sp_cross, m_sp_tpsl]
])
display(metrics_sp.style
    .background_gradient(cmap='RdYlGn', subset=['Total Ret %', 'Sharpe'])
    .format({c: '{:.1f}' for c in ['Total Ret %', 'B&H Ret %', 'CAGR %', 'Max DD %', 'Avg Trade %']}))


,Exit Strategy,Trades,Win Rate %,Avg Trade %,Total Ret %,B&H Ret %,CAGR %,Sharpe,Max DD %,Avg Hold (d)
0,Death Cross,10,70.000000,15.3,268.7,374.0,7.0,0.840000,-8.4,534.000000
1,TP 15% / SL 5%,10,70.000000,8.5,118.1,374.0,4.1,1.160000,-5.0,170.000000


In [ ]:
# ── Equity curve comparison ──────────────────────────────────────
def plot_equity_comparison(df, metrics_list, symbol):
    """
    2-panel chart: Price + MAs (top), Equity curves (bottom).

    FIX: shared_xaxes=True (was False).
    Both panels now synchronise when zooming or panning.
    """
    short_w  = df.attrs.get('short_w', 50)
    long_w   = df.attrs.get('long_w',  200)
    bh_daily = (df['close'] / df['close'].iloc[0] - 1) * 100

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        row_heights=[0.60, 0.40],
        vertical_spacing=0.06,
        subplot_titles=(
            f'{symbol} — Price  |  MA{short_w} vs MA{long_w}',
            'Cumulative Return (%)'
        )
    )

    fig.add_trace(go.Candlestick(
        x=df['datetime'], open=df['open'], high=df['high'],
        low=df['low'], close=df['close'], name='Price',
        increasing_line_color='#26a69a', decreasing_line_color='#ef5350'
    ), row=1, col=1)
    fig.add_trace(go.Scatter(x=df['datetime'], y=df[f'ma{short_w}'],
        line=dict(color='orange', width=1.5), name=f'MA {short_w}'), row=1, col=1)
    fig.add_trace(go.Scatter(x=df['datetime'], y=df[f'ma{long_w}'],
        line=dict(color='royalblue', width=1.5), name=f'MA {long_w}'), row=1, col=1)

    gc_df = df[df['golden_cross']]
    fig.add_trace(go.Scatter(x=gc_df['datetime'], y=gc_df['close'], mode='markers',
        marker=dict(symbol='triangle-up', size=12, color='gold',
                    line=dict(color='black', width=1)), name='Golden Cross ▲'), row=1, col=1)
    dc_df = df[df['death_cross']]
    fig.add_trace(go.Scatter(x=dc_df['datetime'], y=dc_df['close'], mode='markers',
        marker=dict(symbol='triangle-down', size=12, color='silver',
                    line=dict(color='red', width=1)), name='Death Cross ▼'), row=1, col=1)

    fig.add_trace(go.Scatter(x=df['datetime'], y=bh_daily,
        line=dict(color='gray', width=1.5, dash='dot'), name='Buy & Hold'), row=2, col=1)

    colors = ['gold', '#00e5ff', '#66bb6a', '#ef5350']
    for m, color in zip(metrics_list, colors):
        eq = (m['equity_curve'] - 1) * 100
        fig.add_trace(go.Scatter(
            x=m['exit_dates'], y=eq, mode='lines+markers',
            line=dict(color=color, width=2.2), marker=dict(size=5),
            name=f"{m['label']}  ({m['total_ret']:+.0f}%)"
        ), row=2, col=1)

    fig.add_hline(y=0, line_dash='dot', line_color='white', row=2, col=1)
    fig.update_layout(
        height=900, template='plotly_dark',
        xaxis_rangeslider_visible=False,
        legend=dict(orientation='h', y=1.02),
        title=dict(text=f'BackTestLAB — {symbol}  |  Golden Cross Strategy',
                   font=dict(size=16, color='gold'))
    )
    fig.update_yaxes(title_text='Price',      row=1, col=1)
    fig.update_yaxes(title_text='Return (%)', row=2, col=1)
    fig.update_xaxes(rangebreaks=[dict(bounds=['sat', 'mon'])],
                     tickformat='%Y', row=1, col=1)
    fig.show()

plot_equity_comparison(df_sp, [m_sp_cross, m_sp_tpsl], SYMBOL_SP)


---
## 🌍 SECTION 3 — Multi-Asset Study (6 Trending Assets)

Same strategy (MA50/200, Death Cross exit **and** TP 15%/SL 5%) across 6 major assets.
Window: **10 years** of daily data.


In [ ]:
# ════════════════════════════════════════════════════════════════
# SECTION 4 — Multi-Asset Study
# ════════════════════════════════════════════════════════════════

ASSETS = {
    # TECH
    "Apple": "AAPL",
    "Microsoft": "MSFT",
    "Nvidia": "NVDA",
    "Alphabet": "GOOGL",
    "Meta": "META",
    "Amazon": "AMZN",
    "Oracle": "ORCL",
    "Salesforce": "CRM",
    "Adobe": "ADBE",
    "Intel": "INTC",
    "AMD": "AMD",
    "Cisco": "CSCO",
    "IBM": "IBM",
    "Qualcomm": "QCOM",
    "ServiceNow": "NOW",

    # CONSUMER
    "Walmart": "WMT",
    "Costco": "COST",
    "Home Depot": "HD",
    "Lowe's": "LOW",
    "Nike": "NKE",
    "Starbucks": "SBUX",
    "McDonald's": "MCD",
    "Disney": "DIS",
    "Target": "TGT",

    # HEALTHCARE
    "Johnson & Johnson": "JNJ",
    "UnitedHealth": "UNH",
    "Abbott": "ABT",
    "Medtronic": "MDT",
    "Thermo Fisher": "TMO",
    "Danaher": "DHR",
    "Intuitive Surgical": "ISRG",
    "Stryker": "SYK",

    # FINANCE
    "Visa": "V",
    "Mastercard": "MA",
    "JPMorgan": "JPM",
    "Bank of America": "BAC",
    "Morgan Stanley": "MS",
    "Goldman Sachs": "GS",
    "American Express": "AXP",

    # INDUSTRIAL / ENERGY
    "Caterpillar": "CAT",
    "John Deere": "DE",
    "General Electric": "GE",
    "Honeywell": "HON",
    "Union Pacific": "UNP",
    "NextEra Energy": "NEE",

    # EXTRA (pour atteindre 50)
    "ExxonMobil": "XOM",
    "Chevron": "CVX",
    "Coca-Cola": "KO",
    "PepsiCo": "PEP",
    "Berkshire Hathaway": "BRK-B"
}


start_10y = (datetime.datetime.now() - datetime.timedelta(days=365*10)).strftime('%Y-%m-%d')

all_results = []
print('Running multi-asset Golden Cross backtest...')
print('=' * 72)

for name, ticker in ASSETS.items():
    print(f'\n  Processing {name} ({ticker})...')
    df_a = getDataFromYahoo(ticker, start_10y, end_date)

    if df_a is None or len(df_a) < 300:
        print('    → Skipped (not enough data)')
        continue

    df_a    = compute_signals(df_a, short_w=50, long_w=200)
    t_cross = detect_golden_cross_trades(df_a)
    t_tpsl  = detect_trades_tp_sl(df_a, tp=0.15, sl=0.05)

    if not t_cross:
        print('    → No trades detected')
        continue

    mc = compute_metrics(t_cross, df_a, 'Death Cross')
    mt = compute_metrics(t_tpsl,  df_a, 'TP 15% / SL 5%')

    for m, exit_type in [(mc, 'Death Cross'), (mt, 'TP 15% / SL 5%')]:
        if not m:
            continue
        all_results.append({
            'Asset':        name,
            'Exit':         exit_type,
            'Trades':       m['n_trades'],
            'Win Rate %':   m['win_rate'],
            'Avg Trade %':  m['avg_trade'],
            'Total Ret %':  m['total_ret'],
            'B&H Ret %':    m['bh_ret'],
            'Alpha vs B&H': round(m['total_ret'] - m['bh_ret'], 1),
            'CAGR %':       m['cagr'],
            'Sharpe':       m['sharpe'],
            'Max DD %':     m['max_dd'],
        })

    print(f'    Cross : {mc["total_ret"]:+.1f}%  vs B&H {mc["bh_ret"]:+.1f}%  | Sharpe {mc["sharpe"]:.2f}')
    print(f'    TP/SL : {mt["total_ret"]:+.1f}%  vs B&H {mt["bh_ret"]:+.1f}%  | Sharpe {mt["sharpe"]:.2f}')

print('\n' + '=' * 72)

Running multi-asset Golden Cross backtest...

  Processing Apple (AAPL)...
    Cross : +250.9%  vs B&H +827.6%  | Sharpe 0.51
    TP/SL : +30.4%  vs B&H +827.6%  | Sharpe 1.34

  Processing Microsoft (MSFT)...
    Cross : +526.6%  vs B&H +528.1%  | Sharpe 0.86
    TP/SL : +66.2%  vs B&H +528.1%  | Sharpe 2.47

  Processing Nvidia (NVDA)...
    Cross : +6547.6%  vs B&H +6753.2%  | Sharpe 1.21
    TP/SL : +19.4%  vs B&H +6753.2%  | Sharpe 1.91

  Processing Alphabet (GOOGL)...
    Cross : +445.8%  vs B&H +657.0%  | Sharpe 1.16
    TP/SL : +57.8%  vs B&H +657.0%  | Sharpe 1.71

  Processing Meta (META)...
    Cross : +342.6%  vs B&H +380.3%  | Sharpe 0.52
    TP/SL : +7.7%  vs B&H +380.3%  | Sharpe 0.36

  Processing Amazon (AMZN)...
    Cross : +235.4%  vs B&H +470.4%  | Sharpe 0.68
    TP/SL : +30.4%  vs B&H +470.4%  | Sharpe 1.04

  Processing Oracle (ORCL)...
    Cross : +157.1%  vs B&H +294.1%  | Sharpe 0.84
    TP/SL : +30.4%  vs B&H +294.1%  | Sharpe 0.97

  Processing Salesforce (

In [ ]:
# ── Results summary ─────────────────────────────────────────────

df_results = pd.DataFrame(all_results)

# Verdict
df_results['Verdict'] = np.where(
    df_results['Alpha vs B&H'] > 0,
    'Outperform',
    'Underperform'
)

total = len(df_results)
outperf = (df_results['Verdict'] == 'Outperform').sum()
underperf = (df_results['Verdict'] == 'Underperform').sum()

pct_out = 100 * outperf / total
pct_under = 100 * underperf / total

print("\n" + "="*60)
print("📊 GLOBAL PERFORMANCE SUMMARY")
print("="*60)

n_assets = df_results['Asset'].nunique()

print(f"\n📊 Number of assets tested: {n_assets}")
print(f"Total strategies tested : {total}")
print(f"Outperform vs B&H      : {outperf} ({pct_out:.1f}%)")
print(f"Underperform vs B&H    : {underperf} ({pct_under:.1f}%)")


📊 GLOBAL PERFORMANCE SUMMARY

📊 Number of assets tested: 50
Total strategies tested : 100
Outperform vs B&H      : 8 (8.0%)
Underperform vs B&H    : 92 (92.0%)


In [ ]:
# ── Grouped bar chart: strategy vs B&H per asset ─────────────────
def plot_multi_asset(df_results):
    exits  = ['Death Cross', 'TP 15% / SL 5%']
    colors = ['gold', '#00e5ff']

    fig = go.Figure()
    for exit_type, color in zip(exits, colors):
        sub = df_results[df_results['Exit'] == exit_type]
        fig.add_trace(go.Bar(
            x=sub['Asset'], y=sub['Total Ret %'],
            name=exit_type, marker_color=color,
            text=[f'{r:+.0f}%' for r in sub['Total Ret %']],
            textposition='outside'
        ))

    sub_bh = df_results[df_results['Exit'] == 'Death Cross']
    fig.add_trace(go.Scatter(
        x=sub_bh['Asset'], y=sub_bh['B&H Ret %'], mode='markers',
        marker=dict(symbol='diamond', size=14, color='white',
                    line=dict(color='gray', width=2)),
        name='Buy & Hold ◆'
    ))

    fig.update_layout(
        barmode='group', height=540, template='plotly_dark',
        title=dict(
            text='BackTestLAB — Golden Cross vs Buy & Hold (10 years)',
            font=dict(size=15, color='gold')
        ),
        yaxis_title='Total Return (%)',
        legend=dict(orientation='h', y=1.06)
    )
    fig.show()

plot_multi_asset(df_results)


---
## 🔄 SECTION 4 — EUR/USD — A Fairer Test Ground

EUR/USD is a **range-bound asset**: no structural uptrend over 20 years.
Buy & Hold return is slightly negative → the Golden Cross now competes on even terms.


In [ ]:
# ════════════════════════════════════════════════════════════════
# SECTION 6 — EUR/USD Backtest (20 years, MA50/200, Death Cross)
# ════════════════════════════════════════════════════════════════

SYMBOL_FX = 'EURUSD=X'

df_fx = getDataFromYahoo(SYMBOL_FX, start_20y, end_date)
df_fx = compute_signals(df_fx, short_w=50, long_w=200)

bh_fx, bh_fx_cagr = compute_bh_return(df_fx)

print(f'✅ {SYMBOL_FX}')
print(f'   Rows   : {len(df_fx)}')
print(f'   Period : {df_fx["datetime"].iloc[0].date()} → {df_fx["datetime"].iloc[-1].date()}')
print(f'   Buy & Hold return : {bh_fx:+.1f}%  (CAGR: {bh_fx_cagr:+.2f}%/yr)')
print(f'   Golden Cross signals : {df_fx["golden_cross"].sum()}')


✅ EURUSD=X
   Rows   : 4984
   Period : 2007-01-19 → 2026-04-10
   Buy & Hold return : -9.6%  (CAGR: -0.52%/yr)
   Golden Cross signals : 15


In [ ]:
# ── Visualise + run backtest ─────────────────────────────────────
plot_signals(df_fx, SYMBOL_FX)

trades_fx_cross = detect_golden_cross_trades(df_fx)
trades_fx_tpsl  = detect_trades_tp_sl(df_fx, tp=0.15, sl=0.05)

m_fx_cross = compute_metrics(trades_fx_cross, df_fx, 'Death Cross')
m_fx_tpsl  = compute_metrics(trades_fx_tpsl,  df_fx, 'TP 15% / SL 5%  (naive)')

display(display_trades(trades_fx_cross, 'EUR/USD — EXIT: Death Cross'))
display(display_trades(trades_fx_tpsl,  'EUR/USD — EXIT: TP 15% / SL 5%'))



── EUR/USD — EXIT: Death Cross ──


,entry_date,entry_price,exit_date,exit_price,pnl_pct,holding_days,outcome
0,2007-01-19,1.296900,2008-09-23,1.469290,+13.29%,613,closed
1,2009-05-27,1.383390,2010-02-09,1.378210,-0.37%,258,closed
2,2010-10-18,1.396430,2011-10-05,1.333010,-4.54%,352,closed
3,2012-10-29,1.292410,2013-05-17,1.288330,-0.32%,200,closed
4,2013-06-19,1.338990,2013-07-05,1.290770,-3.60%,16,closed
5,2013-07-26,1.327420,2014-07-08,1.360620,+2.50%,347,closed
6,2015-10-05,1.122700,2015-11-26,1.062500,-5.36%,52,closed
7,2016-03-23,1.122300,2016-10-25,1.087550,-3.10%,216,closed
8,2017-05-24,1.117930,2018-06-08,1.179520,+5.51%,380,closed
9,2020-06-29,1.122590,2021-05-06,1.200800,+6.97%,311,closed



── EUR/USD — EXIT: TP 15% / SL 5% ──


,entry_date,entry_price,exit_date,exit_price,pnl_pct,holding_days,outcome
0,2007-01-19,1.296900,2007-11-23,1.491430,+15.00%,308,TP +15%
1,2009-05-27,1.383390,2010-04-28,1.314220,-5.00%,336,SL -5%
2,2010-10-18,1.396430,2010-11-26,1.326610,-5.00%,39,SL -5%
3,2012-10-29,1.292410,2014-12-05,1.227790,-5.00%,767,SL -5%
4,2015-10-05,1.122700,2015-11-17,1.066560,-5.00%,43,SL -5%
5,2016-03-23,1.122300,2016-11-18,1.066180,-5.00%,240,SL -5%
6,2017-05-24,1.117930,2022-04-27,1.062030,-5.00%,1799,SL -5%
7,2022-12-30,1.066080,2026-04-10,1.172880,+10.02%,1197,EOD


In [ ]:
# ── Metrics ──────────────────────────────────────────────────────
metrics_fx = pd.DataFrame([
    {
        'Exit Strategy': m['label'],
        'Trades':        m['n_trades'],
        'Win Rate %':    m['win_rate'],
        'Avg Trade %':   m['avg_trade'],
        'Total Ret %':   m['total_ret'],
        'B&H Ret %':     m['bh_ret'],
        'Alpha vs B&H':  round(m['total_ret'] - m['bh_ret'], 1),
        'CAGR %':        m['cagr'],
        'Sharpe':        m['sharpe'],
        'Max DD %':      m['max_dd'],
    }
    for m in [m_fx_cross, m_fx_tpsl]
])
display(metrics_fx.style
    .background_gradient(cmap='RdYlGn', subset=['Total Ret %', 'Sharpe'])
    .format({c: '{:.1f}' for c in ['Total Ret %', 'B&H Ret %', 'CAGR %',
                                    'Max DD %', 'Avg Trade %', 'Alpha vs B&H']}))


,Exit Strategy,Trades,Win Rate %,Avg Trade %,Total Ret %,B&H Ret %,Alpha vs B&H,CAGR %,Sharpe,Max DD %
0,Death Cross,15,33.300000,0.2,1.0,-9.6,10.6,0.1,0.040000,-14.1
1,TP 15% / SL 5% (naive),8,25.000000,-0.6,-7.0,-9.6,2.6,-0.4,-0.080000,-26.5


---
## 🔬 SECTION 5 — MA Optimisation on EUR/USD

Is MA50/200 the best combination for this asset?
We test 4 classic MA combinations with both the Death Cross exit and the optimised TP/SL.


In [ ]:
# ════════════════════════════════════════════════════════════════
# SECTION 8 — MA Optimisation (EUR/USD)
# ════════════════════════════════════════════════════════════════

best_tp = 0.05
best_sl = 0.02
MA_COMBOS = [(20, 50), (50, 100), (50, 200), (100, 200)]
df_opt_raw = getDataFromYahoo(SYMBOL_FX, start_20y, end_date)

opt_rows = []
print(f'Running MA optimisation on {SYMBOL_FX} ...')
print('─' * 60)

for short_w, long_w in MA_COMBOS:
    df_combo = compute_signals(df_opt_raw.copy(), short_w, long_w)

    for exit_label, tp_val, sl_val in [
        ('Cross',                  None,     None),
        (f'TP {int(best_tp*100)}%/SL {int(best_sl*100)}%', best_tp, best_sl),
    ]:
        if exit_label == 'Cross':
            trades = detect_golden_cross_trades(df_combo)
        else:
            trades = detect_trades_tp_sl(df_combo, tp=tp_val, sl=sl_val)

        if not trades:
            continue

        m = compute_metrics(trades, df_combo, exit_label)
        opt_rows.append({
            'MA Combo':    f'{short_w}/{long_w}',
            'Strategy':    exit_label,
            'Trades':      m['n_trades'],
            'Win Rate %':  m['win_rate'],
            'Avg Trade %': m['avg_trade'],
            'Total Ret %': m['total_ret'],
            'B&H Ret %':   m['bh_ret'],
            'CAGR %':      m['cagr'],
            'Sharpe':      m['sharpe'],
            'Max DD %':    m['max_dd'],
            'Avg Hold (d)': m['avg_holding'],
        })

    print(f'  MA {short_w}/{long_w} done')

df_ma_results = pd.DataFrame(opt_rows)
print()
display(df_ma_results.style
    .background_gradient(cmap='RdYlGn', subset=['Total Ret %', 'Sharpe'])
    .format({
        'Total Ret %': '{:.1f}', 'B&H Ret %': '{:.1f}',
        'CAGR %': '{:.1f}',     'Max DD %':  '{:.1f}',
        'Avg Trade %': '{:.1f}',
    }))


Running MA optimisation on EURUSD=X ...
────────────────────────────────────────────────────────────
  MA 20/50 done
  MA 50/100 done
  MA 50/200 done
  MA 100/200 done



,MA Combo,Strategy,Trades,Win Rate %,Avg Trade %,Total Ret %,B&H Ret %,CAGR %,Sharpe,Max DD %,Avg Hold (d)
0,20/50,Cross,57,35.100000,0.2,6.2,-6.3,0.3,0.100000,-18.4,64.000000
1,20/50,TP 5%/SL 2%,45,28.900000,0.0,-1.2,-6.3,-0.1,0.010000,-26.0,61.000000
2,50/100,Cross,25,44.000000,1.0,23.7,-8.6,1.1,0.270000,-9.1,151.000000
3,50/100,TP 5%/SL 2%,24,37.500000,0.6,14.6,-8.6,0.7,0.390000,-8.8,56.000000
4,50/200,Cross,15,33.300000,0.2,1.0,-9.6,0.1,0.040000,-14.1,242.000000
5,50/200,TP 5%/SL 2%,15,46.700000,1.3,19.7,-9.6,0.9,0.670000,-5.9,73.000000
6,100/200,Cross,13,38.500000,-0.4,-5.6,-9.6,-0.3,-0.110000,-11.9,279.000000
7,100/200,TP 5%/SL 2%,13,38.500000,0.7,8.6,-9.6,0.4,0.350000,-9.6,86.000000


In [ ]:
# ── Bar chart: Death Cross only — clean MA comparison ────────────
#
# FIX: Original chart applied 4 fixed colours to 8+ rows,
# causing wrong colour-to-row assignment.
# Now shows only the Death Cross rows for a clean apples-to-apples
# MA comparison, with colour driven by positive/negative return.
#
df_cross_only = df_ma_results[df_ma_results['Strategy'] == 'Cross'].copy()
bh_val        = df_cross_only['B&H Ret %'].iloc[0]

fig_ma = go.Figure(go.Bar(
    x=df_cross_only['MA Combo'],
    y=df_cross_only['Total Ret %'],
    marker_color=['#66bb6a' if r >= 0 else '#ef5350'
                  for r in df_cross_only['Total Ret %']],
    text=[f'{r:+.1f}%' for r in df_cross_only['Total Ret %']],
    textposition='outside'
))
fig_ma.add_hline(y=bh_val, line_dash='dot', line_color='white',
                 annotation_text=f'Buy & Hold {bh_val:+.0f}%',
                 annotation_position='top right')
fig_ma.update_layout(
    height=460, template='plotly_dark',
    title=dict(text=f'BackTestLAB — MA Optimisation on {SYMBOL_FX} (Death Cross exit)',
               font=dict(size=15, color='gold')),
    yaxis_title='Total Return (%)',
    xaxis_title='MA Combination (short / long)',
    showlegend=False
)
fig_ma.show()


---
## 🏁 SECTION 6 — Scientific Verdict

In [ ]:
# ════════════════════════════════════════════════════════════════
# SECTION 7 — Scientific Verdict
#
# ════════════════════════════════════════════════════════════════

# Best strategy found: MA50/100 Death Cross on EUR/USD
best_ma_row = df_ma_results[
    (df_ma_results['Strategy'] == 'Cross') &
    (df_ma_results['MA Combo']  == '50/100')
].iloc[0]

total_tests        = len(df_results)
beat_bh_fx         = m_fx_cross['total_ret'] > bh_fx

print (best_ma_row)


print('\n' + '═' * 64)
print('          BackTestLAB — SCIENTIFIC VERDICT')
print('═' * 64)

print(f'''
  ── PART 1 : Trending Assets (6 stocks/indices, 10 years) ──
  Strategy : Golden Cross MA50/200  (Death Cross + TP/SL 15/5)
  Result   : {pct_under}/{total_tests} strategies underperform Buy & Hold
  Why      : Structural bull market — B&H mechanically wins


  ── PART 2 : Range-Bound Asset (EUR/USD, 20 years) ──────────
  Strategy : Golden Cross MA{best_ma_row["MA Combo"]} — {best_ma_row["Strategy"]} exit
  ─────────────────────────────────────────────────────────────
  Golden Cross return  : {best_ma_row["Total Ret %"]:+.1f}%
  Buy & Hold return    : {bh_fx:+.1f}%
  Alpha vs B&H         : {best_ma_row["Total Ret %"] - bh_fx:+.1f}%
  Sharpe               : {best_ma_row["Sharpe"]:.2f}
  Max Drawdown         : {best_ma_row["Max DD %"]:.1f}%
  Beats Buy & Hold ?   : {"✅ YES" if beat_bh_fx else "❌ NO  (but B&H also lost money)"}
  ─────────────────────────────────────────────────────────────
  Best MA combo        : {best_ma_row["MA Combo"]}
  ─────────────────────────────────────────────────────────────

  CONCLUSION :
  The Golden Cross is not a bad strategy — it is a misunderstood one.

  It only makes sense when THREE conditions are met:
    1. The right asset   — range-bound, no strong secular trend
    2. The right exit    — TP/SL calibrated to the asset volatility
    3. The right MA      — windows matched to the asset cycle length

  "The entry tells you WHEN the trend starts.
   The asset, the exit, and the parameters determine
   whether you profit from it."
''')

print('═' * 64)


MA Combo        50/100
Strategy         Cross
Trades              25
Win Rate %        44.0
Avg Trade %       0.96
Total Ret %       23.7
B&H Ret %         -8.6
CAGR %            1.09
Sharpe            0.27
Max DD %          -9.1
Avg Hold (d)     151.0
Name: 2, dtype: object

════════════════════════════════════════════════════════════════
          BackTestLAB — SCIENTIFIC VERDICT
════════════════════════════════════════════════════════════════

  ── PART 1 : Trending Assets (6 stocks/indices, 10 years) ──
  Strategy : Golden Cross MA50/200  (Death Cross + TP/SL 15/5)
  Result   : 92.0/100 strategies underperform Buy & Hold
  Why      : Structural bull market — B&H mechanically wins


  ── PART 2 : Range-Bound Asset (EUR/USD, 20 years) ──────────
  Strategy : Golden Cross MA50/100 — Cross exit
  ─────────────────────────────────────────────────────────────
  Golden Cross return  : +23.7%
  Buy & Hold return    : -9.6%
  Alpha vs B&H         : +33.3%
  Sharpe               : 0.27
  Max